## Importaciones de Librerías

In [30]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from lightgbm import LGBMClassifier

import optuna

import joblib

import requests

from dotenv import load_dotenv
import os

## Carga de Datos

In [31]:
load_dotenv()
URL_BACKEND_INTERNSHIPS = os.getenv('URL_BACKEND_INTERNSHIPS')

In [32]:
def get_internships():
    try:
        response = requests.get(URL_BACKEND_INTERNSHIPS)
        if response.status_code == 200:
            return response.json()
        else:
            print(f'Error al obtener los datos: {response.status_code} - {response.text}')
            return None
    except Exception as e:
        print(f'Error con la conexión a la API de las prácticas: {e}')
        return None

In [33]:
df = pd.DataFrame(get_internships())
print("Datos cargados exitosamente")

Datos cargados exitosamente


## Preparación Varibales x, y

In [34]:
# Definimos solo las variables que demostraron verdadero valor predictivo
variables_seleccionadas = [
    'interview_score', 'skills_score', 'communication_score', 'coding_test_score',
    'projects_count', 'soft_skills_score', 'certifications_count', 'resume_score',
    'internships_done', 'extracurricular', 'consistency_score', 'placement_training',
    'cgpa', 'github_score'
]

X = df[variables_seleccionadas]
y = df['selected']

## Separación varibales en entreno y test

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42, stratify=y)

## Búsuqueda de Mejores Hiperparametros con Optuna

In [36]:
def objective_lightgbm(trial):
    params = {
        'boosting_type': trial.suggest_categorical('boosting_type', ['rf', 'gbdt']),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'num_leaves': trial.suggest_int('max_leaves', 10, 35),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'bagging_freq': 1,
        'bagging_fraction': 0.8,
        'feature_fraction': 0.8,
        'objective': 'binary',
        'random_state': 42
    }

    model = LGBMClassifier(**params)
    return cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc').mean()
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lightgbm, n_trials=25)
print('LightGBM - Mejores Hiperparámetros: ', study_lgb.best_params)

[I 2026-06-24 18:05:10,102] A new study created in memory with name: no-name-809e8fa2-1f4f-45cc-b0c8-30dd3da99f13


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Number of positive: 4719, number of negative: 1681
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.036516 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 363
[LightGBM] [Info] Number of data point

[I 2026-06-24 18:05:16,832] Trial 0 finished with value: 0.5642133243553162 and parameters: {'boosting_type': 'rf', 'learning_rate': 0.07055409748865604, 'max_leaves': 15, 'max_depth': 15}. Best is trial 0 with value: 0.5642133243553162.


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Number of positive: 4719, number of negative: 1681
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001451 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 363
[LightGBM] [Info] Number of data point

[I 2026-06-24 18:05:17,693] Trial 1 finished with value: 0.5717494390763711 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.06656596688304325, 'max_leaves': 11, 'max_depth': 5}. Best is trial 1 with value: 0.5717494390763711.


...

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 

[I 2026-06-24 18:05:36,186] Trial 23 finished with value: 0.5732310413389137 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.029817096627879035, 'max_leaves': 10, 'max_depth': 14}. Best is trial 12 with value: 0.5751780315297156.


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 

[I 2026-06-24 18:05:36,874] Trial 24 finished with value: 0.5698128396406361 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.010716602235386057, 'max_leaves': 13, 'max_depth': 7}. Best is trial 12 with value: 0.5751780315297156.


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
LightGBM - Mejores Hiperparámetros:  {'boosting_type': 'gbdt', 'learning_rate': 0.07084738752475225, 'max_leaves': 10, 'max_depth': 11}


## Pipeline

In [38]:
pipeline = Pipeline(
    steps = [
        ('model', LGBMClassifier(**study_lgb.best_params, eval_metric='logloss', random_state=42))
    ]
)

In [39]:
print("Entrenando el modelo final con los mejores parámetros...")
pipeline.fit(X_train, y_train)

Entrenando el modelo final con los mejores parámetros...
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] num_leaves is set=31, max_leaves=10 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] num_leaves is set=31, max_leaves=10 will be ignored. Current value: num_leaves=31
[LightGBM] [Info] Number of positive: 5899, number of negative: 2101
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001450 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 363
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.737375 -> initscore=1.032369
[LightGBM] [Info] Start training from score 1.032369


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[<U20](14,)","['interview_score','skills_score','communication_score',..., 'placement_training','cgpa','github_score']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,max_depth,11
,learning_rate,0.07084738752475225
,random_state,42


In [40]:
# Calcular predicciones para ambos conjuntos
y_train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
y_test_pred_proba = pipeline.predict_proba(X_test)[:, 1]

# Calcular el ROC-AUC para ambos
auc_train = roc_auc_score(y_train, y_train_pred_proba)
auc_test = roc_auc_score(y_test, y_test_pred_proba)

print(f"ROC-AUC Entrenamiento: {auc_train:.4f}")
print(f"ROC-AUC Prueba (Test):  {auc_test:.4f}")

# Calcular la brecha (gap)
brecha = auc_train - auc_test
print(f"Brecha de Overfitting:   {brecha:.4f}")

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] num_leaves is set=31, max_leaves=10 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] num_leaves is set=31, max_leaves=10 will be ignored. Current value: num_leaves=31
ROC-AUC Entrenamiento: 0.9014
ROC-AUC Prueba (Test):  0.5362
Brecha de Overfitting:   0.3652


In [41]:
print("\nEvaluando el modelo en el conjunto de prueba (X_test)...")
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]


Evaluando el modelo en el conjunto de prueba (X_test)...
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] num_leaves is set=31, max_leaves=10 will be ignored. Current value: num_leaves=31
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] num_leaves is set=31, max_leaves=10 will be ignored. Current value: num_leaves=31


In [42]:
print("\n=== Reporte de Clasificación ===")
print(classification_report(y_test, y_pred))

print("=== Matriz de Confusión ===")
print(confusion_matrix(y_test, y_pred))

print(f"\nROC-AUC Score en Test: {roc_auc_score(y_test, y_pred_proba):.4f}")


=== Reporte de Clasificación ===
              precision    recall  f1-score   support

           0       0.45      0.02      0.04       525
           1       0.74      0.99      0.85      1475

    accuracy                           0.74      2000
   macro avg       0.60      0.51      0.44      2000
weighted avg       0.66      0.74      0.63      2000

=== Matriz de Confusión ===
[[  10  515]
 [  12 1463]]

ROC-AUC Score en Test: 0.5362


## Guardado del modelo

In [43]:
joblib.dump(pipeline, '../models/lgb_model.pkl')

['../models/lgb_model.pkl']